### load the raw data

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

raw = Path("../data/raw")
processed = Path("../data/processed")
processed.mkdir(exist_ok=True)

names = ["website_sessions", "website_pageviews", "orders_mav",
         "order_items", "order_item_refunds", "products"]
data = {n: pd.read_csv(raw / f"{n}.csv") for n in names}

### 2: turn the text dates into real dates

In [5]:
for n, df in data.items():
    df["created_at"] = pd.to_datetime(df["created_at"], format="%m/%d/%Y %H:%M")
    print(n, df["created_at"].dtype)

website_sessions datetime64[us]
website_pageviews datetime64[us]
orders_mav datetime64[us]
order_items datetime64[us]
order_item_refunds datetime64[us]
products datetime64[us]


### 3: create the marketing channel column

In [6]:
s = data["website_sessions"]
s["marketing_channel"] = np.where(
    s["utm_source"].notna(), s["utm_source"],
    np.where(s["http_referer"].notna(), "organic_search", "direct")
)
print(s["marketing_channel"].value_counts())

marketing_channel
gsearch           316035
bsearch            62823
organic_search     43411
direct             39917
socialbook         10685
Name: count, dtype: int64


### 4: save the cleaned files

In [7]:
for n, df in data.items():
    df.to_csv(processed / f"{n}_clean.csv", index=False)
print("saved")

saved


### 5: verify everything

In [8]:
s, pv = data["website_sessions"], data["website_pageviews"]
o, oi = data["orders_mav"], data["order_items"]
r, p = data["order_item_refunds"], data["products"]

print("DATE RANGES")
for n, df in data.items():
    print(f"  {n}: {df['created_at'].min()} to {df['created_at'].max()}")

print("\nBROKEN LINKS (should all be 0)")
print("  orders with no session:", (~o["website_session_id"].isin(s["website_session_id"])).sum())
print("  order_items with no order:", (~oi["order_id"].isin(o["order_id"])).sum())
print("  order_items with no product:", (~oi["product_id"].isin(p["product_id"])).sum())
print("  refunds with no order_item:", (~r["order_item_id"].isin(oi["order_item_id"])).sum())
print("  pageviews with no session:", (~pv["website_session_id"].isin(s["website_session_id"])).sum())

print("\nMONEY CHECK")
print("  revenue in orders_mav :", round(o["price_usd"].sum(), 2))
print("  revenue in order_items:", round(oi["price_usd"].sum(), 2))
print("  total refunds         :", round(r["refund_amount_usd"].sum(), 2))

DATE RANGES
  website_sessions: 2012-03-19 08:04:00 to 2015-03-19 07:59:00
  website_pageviews: 2012-03-19 08:04:00 to 2015-01-16 19:03:00
  orders_mav: 2012-03-19 10:42:00 to 2015-03-19 05:38:00
  order_items: 2012-03-19 10:42:00 to 2015-03-19 05:38:00
  order_item_refunds: 2012-04-06 11:32:00 to 2015-04-01 18:11:00
  products: 2012-03-19 08:00:00 to 2014-02-05 10:00:00

BROKEN LINKS (should all be 0)
  orders with no session: 0
  order_items with no order: 0
  order_items with no product: 0
  refunds with no order_item: 0
  pageviews with no session: 0

MONEY CHECK
  revenue in orders_mav : 1938509.75
  revenue in order_items: 1938509.75
  total refunds         : 85338.69
